# 3. One catalog table and a keyword lexicon

I work with two catalogs of Spanish-language printed editions:

- **USTC** (Universal Short Title Catalogue), 1500–1700, which I download in notebook 1. It lists no Spanish editions after 1700.
- **BNE** (Biblioteca Nacional de España), catalog of early printed books ("monografías antiguas"), 1501–1800, which I downloaded
  by hand from datosabiertos.bne.es (the server refuses scripted downloads).

Here I put both in one table (one row per catalog record), add length and Jesuit authorship, and mark each title with a keyword
lexicon (`lexicon.py`): **spiritual**, **other pastoral** (sermons, sacred history, moral theology, catechisms) or **none**.
The lexicon is only a first pass: in notebook 6 I draw a sample stratified by these three classes, and in notebook 7 I classify
it with an open language model.

The two catalogs overlap (I link them in notebook 4), so the table counts catalog records, not distinct editions.

---

# 3. Una tabla con los dos catálogos y un léxico de palabras clave

Trabajo con dos catálogos de impresos en castellano:

- **USTC** (Universal Short Title Catalogue), 1500–1700, que descargo en el notebook 1. No registra ediciones en castellano
  después de 1700.
- **BNE** (Biblioteca Nacional de España), catálogo de "monografías antiguas", 1501–1800, que bajé a mano de
  datosabiertos.bne.es (el servidor rechaza las descargas automáticas).

Acá junto los dos en una tabla (una fila por registro de catálogo), agrego la extensión y la autoría jesuita, y marco cada título
con un léxico de palabras clave (`lexicon.py`): **espiritual**, **otra pastoral** (sermones, historia sacra, teología moral,
catecismos) o **ninguna**. El léxico es solo una primera pasada: en el notebook 6 saco una muestra estratificada por estas tres
clases, y en el notebook 7 la clasifico con un modelo de lenguaje abierto.

Los dos catálogos se superponen (los enlazo en el notebook 4), así que la tabla cuenta registros de catálogo, no ediciones
distintas.

In [1]:
# Celda 1
# Paths and parameters:
%cd "E:/No sincroniza/Software desarrollado/spanish-imprints-counts"
DATA = "E:/Datos/spanish_imprints"
USTC = DATA + "/ustc_spanish.jsonl"                 # From notebook 1.
BNE_CATALOG = DATA + "/bne/monoantiguas-UTF8.zip"   # Downloaded by hand.
JESUITS = DATA + "/jesuit_authors.tsv"              # From notebook 2.
OUT = DATA + "/catalog.tsv"                         # One row per record (stays outside the repository).
YEAR_FROM, YEAR_TO = 1501, 1800
LANGUAGE = "español"

E:\No sincroniza\Software desarrollado\spanish-imprints-counts


In [2]:
# Celda 2
import re, zipfile
import pandas as pd
from helpers import norm, name_key, ustc_pages, bne_pages, volumes, SJ_TITLE, SJ_CORPORATE, NOT_JESUIT_KEYS
from lexicon import match_groups, spiritual_author

## 1. How many USTC editions are there, how long are they, and how many are by Jesuits? / ¿Cuántas ediciones tiene el USTC, qué extensión tienen y cuántas son de jesuitas?

In [3]:
# Celda 3
# One line per (edition, subject) in the download: an edition filed under two subjects appears twice:
raw = pd.read_json(USTC, lines=True, dtype={"id": str, "sn": str})
ustc = raw.drop_duplicates("id").copy()
print("Lines:", len(raw), "| editions:", len(ustc))
ustc[["id", "year", "title", "authorship", "heading", "pagination"]].head()

Lines: 97832 | editions: 82768


,id,year,title,authorship,heading,pagination
0,100000,1567,Luz del alma christiana contra la ceguedad e i...,"Meneses, Felipe de",None,ff. cli [1].
1,25682,1640,Nos don Cesar Facheneti nuncio y colector gene...,None,"Ecclesia Catholica - Nuncio - Facheneti, César",ff. 3 [1].
2,25710,1621,"Sermon que predico fray Francisco de Biuero, d...","Bivero, Francisco",None,pp. [4] 46 [2]
3,32029,1588,Los libros de la madre Teresa de Jesus,"Avila, St Teresa of",None,pp. [8] 560 [8] 268 304.
4,32121,1650,Historia y vida de la venerable madre Angela M...,"Fons, Juan Pablo",None,ff. [13] 252 [=253] [41].


In [4]:
# Celda 4
ustc["pages"] = ustc["pagination"].map(ustc_pages)
print("Without a page count:", ustc["pages"].isna().sum(), "of", len(ustc))
ustc["pages"].describe()

Without a page count: 14838 of 82768


count    67930.000000
mean       158.342735
std        306.784990
min          0.000000
25%          8.000000
50%         28.000000
75%        160.000000
max      14769.000000
Name: pages, dtype: float64

**Jesuit authors.** Strict: the personal author (`authorship`) is in the list of notebook 2. Broad: strict, or the title says
"por el padre … de la Compañía de Jesús" (this also catches translators). I flag apart the Jesuit works without a personal author
("Institutum Societatis Iesu", heading "Jesuitas"), and I exclude known homonyms (`helpers.NOT_JESUIT`).

---

**Autores jesuitas.** Criterio estricto: el autor personal (`authorship`) está en la lista del notebook 2. Criterio amplio: el
estricto, o el título dice "por el padre … de la Compañía de Jesús" (esto también atrapa a traductores). Marco aparte las obras
jesuitas sin autor personal ("Institutum Societatis Iesu", encabezamiento "Jesuitas") y excluyo los homónimos conocidos
(`helpers.NOT_JESUIT`).

In [5]:
# Celda 5
jesuit_keys = set(pd.read_csv(JESUITS, sep="\t", dtype=str)["key"])
ustc["author"] = ustc["authorship"].fillna("")
ustc["has_author"] = ustc["author"] != ""
ustc["author_key"] = ustc["author"].map(lambda a: " ".join(sorted(name_key(a))))
not_jesuit = ustc["author_key"].isin(NOT_JESUIT_KEYS)
ustc["jesuit_strict"] = ustc["has_author"] & ustc["author_key"].isin(jesuit_keys) & ~not_jesuit
title_rule = ustc["title"].map(norm).str.contains(SJ_TITLE)
title_authors = set(ustc.loc[title_rule & ustc["has_author"] & ~not_jesuit, "author"])
ustc["jesuit_broad"] = ustc["jesuit_strict"] | ustc["author"].isin(title_authors) | title_rule
ustc["jesuit_corporate"] = ~ustc["has_author"] & (ustc["heading"].map(norm).str.contains(SJ_CORPORATE) | ustc["title"].map(norm).str.contains(SJ_CORPORATE))
ustc[["has_author", "jesuit_strict", "jesuit_broad", "jesuit_corporate"]].sum()

has_author          42308
jesuit_strict        3690
jesuit_broad         5254
jesuit_corporate      752
dtype: int64

## 2. And in the BNE? / ¿Y en la BNE?

In [6]:
# Celda 6
catalog = pd.read_csv(zipfile.ZipFile(BNE_CATALOG).open("moa/monoantiguas-UTF8.csv"), sep=";", dtype=str)
catalog["year"] = pd.to_numeric(catalog["fecha_de_publicacion"], errors="coerce")
bne = catalog[(catalog["year"] >= YEAR_FROM) & (catalog["year"] <= YEAR_TO) & (catalog["lengua_principal"].str.strip() == LANGUAGE)].copy()
bne["id"] = bne["id"].str.strip()
print("All records:", len(catalog), f"| Spanish-language, {YEAR_FROM}–{YEAR_TO}:", len(bne))

All records: 218481 | Spanish-language, 1501–1800: 80401


In [7]:
# Celda 7
bne["pages"] = bne["extension"].map(bne_pages)
bne["volumes"] = bne["extension"].map(volumes)
print("Without a page count:", bne["pages"].isna().sum(), "| multi-volume:", (bne["volumes"] > 1).sum())
bne["pages"].describe()

Without a page count: 2201 | multi-volume: 2202


count    78200.000000
mean       155.149437
std        369.556066
min          0.000000
25%          8.000000
50%         30.000000
75%        120.000000
max      19010.000000
Name: pages, dtype: float64

**Authors.** `autores` lists people, some with a role: "Pelfort, Jaime( autor) /**/ Suriá y Burgada, Francisco, ( fl. 1771-1805)( impresor)".
I take as personal author the first person with no role or with the role "autor". The BNE often adds the order to the name:
"(S.I.)" marks Jesuits. I flag Jesuits as for the USTC, and I also read the statement of responsibility (`mencion_de_autores`).

---

**Autores.** `autores` enumera personas, algunas con su función: "Pelfort, Jaime( autor) /**/ Suriá y Burgada, Francisco, ( fl. 1771-1805)( impresor)".
Tomo como autor personal la primera persona sin función o con la función "autor". La BNE suele agregar la orden al nombre:
"(S.I.)" marca a los jesuitas. Marco a los jesuitas como en el USTC y además leo la mención de responsabilidad
(`mencion_de_autores`).

In [8]:
# Celda 8
# "Salas, Pedro de, ( 1584-1664)" -> person; "Valdés, Alejandro( impresor)" -> skipped:
def first_author_entry(autores):
    if not isinstance(autores, str):
        return ""
    for person in autores.split("/**/"):
        roles = [r.strip() for r in re.findall(r"\(\s*([a-záéíóú ]+)\)", person)]
        if not roles or "autor" in roles or "autor atribuido" in roles:
            return person.strip()
    return ""

author_entry = bne["autores"].map(first_author_entry)
bne["author"] = author_entry.str.split("(").str[0].str.strip(" ,")
bne["has_author"] = bne["author"] != ""
bne["author_key"] = bne["author"].map(lambda a: " ".join(sorted(name_key(a))))
not_jesuit = bne["author_key"].isin(NOT_JESUIT_KEYS)
sj_mark = author_entry.str.contains(r"\(\s*S\.\s?I\.\s*\)")
bne["jesuit_strict"] = bne["has_author"] & (bne["author_key"].isin(jesuit_keys) | sj_mark) & ~not_jesuit
statement = (bne["mencion_de_autores"].fillna("") + " " + bne["titulo"].fillna("")).map(norm)
statement_rule = statement.str.contains(SJ_TITLE) & ~not_jesuit
statement_authors = set(bne.loc[statement_rule & bne["has_author"], "author"])
bne["jesuit_broad"] = bne["jesuit_strict"] | statement_rule | bne["author"].isin(statement_authors)
corporate_text = (bne["tema"].fillna("") + " " + bne["titulo"].fillna("") + " " + bne["mencion_de_autores"].fillna("")).map(norm)
bne["jesuit_corporate"] = ~bne["has_author"] & corporate_text.str.contains(SJ_CORPORATE)
bne[["has_author", "jesuit_strict", "jesuit_broad", "jesuit_corporate"]].sum()

has_author          49451
jesuit_strict        3014
jesuit_broad         4131
jesuit_corporate      759
dtype: int64

## 3. How do I put both catalogs in one table? / ¿Cómo junto los dos catálogos en una tabla?

`author_text` keeps the whole author field (all names in the BNE), which the lexicon's author rule reads; `author` is the
personal author only.

---

`author_text` guarda el campo de autor completo (todos los nombres, en la BNE), que es lo que lee la regla de autores del
léxico; `author` es solo el autor personal.

In [9]:
# Celda 9
columns = ["source", "id", "year", "title", "author_text", "author", "has_author", "pages", "volumes",
           "jesuit_strict", "jesuit_broad", "jesuit_corporate"]
ustc_table = ustc.assign(source="ustc", author_text=ustc["authorship"].fillna(""), volumes=1)
bne_table = bne.assign(source="bne", title=bne["titulo"], author_text=bne["autores"].fillna(""))
table = pd.concat([ustc_table[columns], bne_table[columns]], ignore_index=True)
table["year"] = pd.to_numeric(table["year"], errors="coerce")
table["decade"] = table["year"] // 10 * 10
print(table.groupby("source").size())
table.head()

source
bne     80401
ustc    82768
dtype: int64


,source,id,year,title,author_text,author,has_author,pages,volumes,jesuit_strict,jesuit_broad,jesuit_corporate,decade
0,ustc,100000,1567.0,Luz del alma christiana contra la ceguedad e i...,"Meneses, Felipe de","Meneses, Felipe de",True,304.0,1,False,False,False,1560.0
1,ustc,25682,1640.0,Nos don Cesar Facheneti nuncio y colector gene...,,,False,8.0,1,False,False,False,1640.0
2,ustc,25710,1621.0,"Sermon que predico fray Francisco de Biuero, d...","Bivero, Francisco","Bivero, Francisco",True,52.0,1,False,False,False,1620.0
3,ustc,32029,1588.0,Los libros de la madre Teresa de Jesus,"Avila, St Teresa of","Avila, St Teresa of",True,1148.0,1,False,False,False,1580.0
4,ustc,32121,1650.0,Historia y vida de la venerable madre Angela M...,"Fons, Juan Pablo","Fons, Juan Pablo",True,1118.0,1,False,False,False,1650.0


## 4. What does the keyword lexicon mark? / ¿Qué marca el léxico de palabras clave?

`lexicon.py` folds old spellings ("christiana" → "cristiana", "deuocion" → "devocion", "exercicios" → "ejercicios") and looks
for terms at the start of words: "sermon" matches "sermones", "meditacion" matches "meditaciones". I assign each term to one
group: spiritual, preaching, sacred history, moral theology or catechism. When several groups match, a fixed order decides
(preaching before spiritual: "Sermón de la Pasión" is a sermon). I give no group to titles that start like lawsuits or decrees
("Por el convento de…", "Nos don…") or contain words such as "villancicos", "pleito" or "comedia".

**Author rule:** when the title matches nothing, I mark the record as spiritual if its author is in a short list of spiritual
authors (Luis de Granada, Kempis, Teresa de Jesús, Juan de Ávila, Luis de la Puente, Nieremberg…): "Granada, Luis de: *Las obras*".

---

`lexicon.py` unifica las grafías antiguas ("christiana" → "cristiana", "deuocion" → "devocion", "exercicios" → "ejercicios") y
busca términos al comienzo de las palabras: "sermon" encuentra "sermones", "meditacion" encuentra "meditaciones". Asigno cada
término a un grupo: espiritual, predicación, historia sacra, teología moral o catecismo. Cuando coinciden varios grupos, decide un
orden fijo (predicación antes que espiritual: "Sermón de la Pasión" es un sermón). No asigno grupo a los títulos que empiezan como
pleitos o edictos ("Por el convento de…", "Nos don…") ni a los que tienen palabras como "villancicos", "pleito" o "comedia".

**Regla de autores:** cuando el título no coincide con nada, marco el registro como espiritual si el autor está en una lista breve
de autores espirituales (Luis de Granada, Kempis, Teresa de Jesús, Juan de Ávila, Luis de la Puente, Nieremberg…):
"Granada, Luis de: *Las obras*".

In [10]:
# Celda 10
# "Sermon que predico..." -> "preaching"; no match -> "none":
table["group"] = table["title"].map(lambda t: (sorted(match_groups(t)) or ["none"])[0])
table["by_author"] = (table["group"] == "none") & table["author_text"].map(spiritual_author)
table["lexicon_class"] = "other_pastoral"
table.loc[table["group"] == "none", "lexicon_class"] = "none"
table.loc[(table["group"] == "spiritual") | table["by_author"], "lexicon_class"] = "spiritual"
print("Marked by the author rule:", table["by_author"].sum())
pd.crosstab(table["lexicon_class"], table["source"], margins=True)

Marked by the author rule: 760


source,bne,ustc,All
lexicon_class,,,
none,66682,65386,132068
other_pastoral,8910,11760,20670
spiritual,4809,5622,10431
All,80401,82768,163169


In [11]:
# Celda 11
# Five titles of each class, to see what the lexicon does:
for lexicon_class, rows in table.groupby("lexicon_class"):
    print("\n" + lexicon_class.upper())
    for title in rows["title"].sample(5, random_state=1):
        print("  ", str(title)[:120])


NONE
   Dialogos militares y politicos discurridos por Eraclito y Democrito sobre las campañas y exercitos de Flandes...:
   Por Iuan Romero Macotela. En respuesta del papel del señor Conde Duque, que responde a la replica de Iuan Romero ... [pl
   Discursos de verdadera cirugía, y censura de ambas vias, y eleccion de la primera intencion curativa y unicion de las he
   Por Martin Sanchez de Aranzamendi. Contra Antonio Lopez Cortizos. A la informacion principal, dada por Martin Sa[n]chez,
   Clausulas de las fundaciones de los mayorazgos que fundaron Christoval Riqvelme, y D. Honofra Riquelme su muger ... ::  

OTHER_PASTORAL
   Segunda cancion, y funebre romance a la vida, muerte, y heroicas virtudes del serenissimo señor don Juan de Austria
   Devota excitación para el incessante reconocimiento y continua gratitud ... á los ... Santos Angeles, y especialmente á 
   Oracion gratulatoria que en la fiesta, que celebro el Real Convento de S. Francisco de Zaragoza en 4 de enero de 1784 :

In [12]:
# Celda 12
table.to_csv(OUT, sep="\t", index=False)
print("Written:", OUT, table.shape)

Written: E:/Datos/spanish_imprints/catalog.tsv (163169, 16)
